In [ ]:
#------------Import Libraries-------------
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from torchvision.models import MobileNet_V2_Weights
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# --------------Device setup-------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ----------Image settings------------
IMG_SIZE = 224
BATCH_SIZE = 16

print("Image Size:", IMG_SIZE)
print("Batch Size:", BATCH_SIZE)

In [ ]:
#--------Data Preprocesing-----
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor()
])

print(transform)

In [ ]:
#----------Load dataset-----------
dataset = datasets.ImageFolder(root="dataset", transform=transform)

print("Classes:", dataset.classes)
print("Total Images:", len(dataset))

In [ ]:
#----------split dataset----------
train_size = int(0.7 * len(dataset))
val_size = int(0.2 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_data, val_data, test_data = random_split(
    dataset, [train_size, val_size, test_size]
)

print("Training Images:", len(train_data))
print("Validation Images:", len(val_data))
print("Test Images:", len(test_data))

In [ ]:
#------------Data loaders------------
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

for images, labels in train_loader:
    print("Image Batch Shape:", images.shape)
    print("Labels:", labels)
    break

In [ ]:
#----------mobilenetv2 model-------------
model = models.mobilenet_v2(
    weights=MobileNet_V2_Weights.DEFAULT
)

# Freeze layers
for param in model.parameters():
    param.requires_grad = False

# Modify classifier
model.classifier[1] = nn.Linear(model.last_channel, 2)

model = model.to(device)

print("Model running on:", device)
print(model.classifier)

In [ ]:
#-----------Loss function & optimizer------
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

print("Loss Function:")
print(criterion)

print("Optimizer:")
print(optimizer)

In [ ]:
#-------------Model Training----------
epochs = 5

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.2f}%")

In [ ]:
#-------Model evaluation--------
model.eval()

all_labels = []
all_preds = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

accuracy = np.mean(np.array(all_labels) == np.array(all_preds))

print("Test Accuracy:", accuracy * 100)

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:", cm)

print(classification_report(all_labels, all_preds))

In [ ]:
#------------Decision fusion  logic-----------
def get_fatigue_level(pred):

    if pred == 0:
        return "Alert"
    else:
        return "Fatigue"

fatigue_levels = [get_fatigue_level(p) for p in all_preds]

print(fatigue_levels[:10])

In [ ]:
#----------Fatigue Progression curve------------
fatigue_stages = []

for p in all_preds:

    if p == 0:
        fatigue_stages.append(0)
    else:
        fatigue_stages.append(2)

interval_size = 30

avg_fatigue = []
time_intervals = []

for i in range(0, len(fatigue_stages), interval_size):

    chunk = fatigue_stages[i:i+interval_size]

    if len(chunk) == 0:
        continue

    avg = sum(chunk) / len(chunk)

    avg_fatigue.append(avg)
    time_intervals.append(i // interval_size)

plt.plot(time_intervals, avg_fatigue, marker='o')
plt.xlabel("Time Interval")
plt.ylabel("Fatigue Level")
plt.title("Driver Fatigue Progression")
plt.show()

In [ ]:
torch.save(model.state_dict(), "model.pth")
print("✅ Model saved successfully")